# Lecture 01a: ML Foundations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_ML_Engineering/blob/main/lectures/01_a_ml_foundations/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_ML_Engineering/main/lectures/01_a_ml_foundations/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import test_vanilla_sgd, test_sgd_momentum, test_adam

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

## The Banana Dataset

Two crescent-shaped clusters ("bananas"). Simple enough to train quickly, but the curved decision boundary makes optimizer differences clearly visible.

In [ ]:
def make_bananas(n=2000, noise=0.15):
    n_top = n // 2
    n_bot = n - n_top
    theta_top = torch.linspace(0, torch.pi, n_top)
    theta_bot = torch.linspace(0, torch.pi, n_bot)
    x_top = torch.stack([torch.cos(theta_top), torch.sin(theta_top)], dim=1)
    x_bot = torch.stack([1 - torch.cos(theta_bot), -torch.sin(theta_bot) + 0.3], dim=1)
    x = torch.cat([x_top, x_bot]) + torch.randn(n, 2) * noise
    y = torch.cat([torch.zeros(n_top), torch.ones(n_bot)])
    perm = torch.randperm(n)
    return x[perm], y[perm]

x_train, y_train = make_bananas(4000)
x_val, y_val = make_bananas(1000)
train_set = TensorDataset(x_train, y_train)

plt.figure(figsize=(5, 5))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap='RdBu', s=3, alpha=0.5)
plt.title('Two Bananas')
plt.axis('equal')
plt.show()

## Model and Training Infrastructure

A feedforward network and training loop. These are provided so you can focus on the optimizers.

In [ ]:
class BananaNet(nn.Module):
    def __init__(self, hidden=(32, 32)):
        super().__init__()
        sizes = [2, *hidden, 1]
        layers = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train(model, optimizer, dataset, epochs=30, batch_size=64):
    loader = DataLoader(dataset, batch_size, shuffle=True)
    losses = []
    for _ in tqdm(range(epochs), leave=False):
        epoch_loss = 0
        for xb, yb in loader:
            loss = F.binary_cross_entropy_with_logits(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb) / len(dataset)
        losses.append(epoch_loss)
    return losses


def accuracy(model, x=x_val, y=y_val):
    with torch.no_grad():
        return ((model(x) > 0).float() == y).float().mean().item()

---

## Exercise: Implement Optimizers from Scratch

Each optimizer is a class with `step()` and `zero_grad()` methods.
The `__init__` and `zero_grad` are provided. **You fill in `step()`.**

After each part, run the test cell to check your implementation against PyTorch's built-in optimizers.

### Part A: Vanilla SGD

The simplest possible optimizer. Each step:

$$\theta_{t+1} = \theta_t - \alpha \, \nabla_\theta \mathcal{L}$$

In [ ]:
class VanillaSGD:
    def __init__(self, params, lr=0.01):
        self.params = list(params)
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for p in self.params:
                # TODO (1 line)
                pass

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
test_vanilla_sgd(VanillaSGD)

<details>
<summary><b>Hint</b></summary>

Subtract the gradient, scaled by the learning rate, from the parameter data.
</details>

<details>
<summary><b>Solution</b></summary>

```python
p.data -= self.lr * p.grad
```
</details>

### Part B: SGD with Momentum

Momentum keeps a running velocity that smooths out noisy gradients and accelerates through flat regions:

$$v_{t+1} = \beta \, v_t + \nabla_\theta \mathcal{L}$$
$$\theta_{t+1} = \theta_t - \alpha \, v_{t+1}$$

Think of a ball rolling downhill: even when the local slope changes direction, it retains some of its previous velocity.

In [ ]:
class SGDMomentum:
    def __init__(self, params, lr=0.01, momentum=0.9):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        with torch.no_grad():
            for i, p in enumerate(self.params):
                # TODO: update self.v[i] and p.data (2 lines)
                pass

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
test_sgd_momentum(SGDMomentum)

<details>
<summary><b>Hint</b></summary>

Update the velocity as a weighted sum of the old velocity and the current gradient. Then update the parameters using the velocity instead of the raw gradient.
</details>

<details>
<summary><b>Solution</b></summary>

```python
self.v[i] = self.momentum * self.v[i] + p.grad
p.data -= self.lr * self.v[i]
```
</details>

### Part C: Adam

Adam combines two ideas:
- **Momentum** (first moment $m$): a running average of the gradient
- **RMSProp** (second moment $v$): a running average of the *squared* gradient, giving a per-parameter learning rate

Both are initialized at zero and thus biased toward zero early in training, so Adam applies a correction:

$$m_{t+1} = \beta_1 m_t + (1 - \beta_1) \, g_t$$
$$v_{t+1} = \beta_2 v_t + (1 - \beta_2) \, g_t^2$$
$$\hat{m} = \frac{m_{t+1}}{1 - \beta_1^{\,t+1}}, \quad \hat{v} = \frac{v_{t+1}}{1 - \beta_2^{\,t+1}}$$
$$\theta_{t+1} = \theta_t - \alpha \, \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}$$

In [ ]:
class Adam:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = [torch.zeros_like(p) for p in self.params]  # first moment
        self.v = [torch.zeros_like(p) for p in self.params]  # second moment
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                g = p.grad
                # TODO: update self.m[i] — biased first moment estimate
                # TODO: update self.v[i] — biased second moment estimate
                # TODO: compute bias-corrected m_hat and v_hat
                # TODO: update p.data
                pass

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
test_adam(Adam)

<details>
<summary><b>Hint</b></summary>

Update the first moment as an exponential moving average of the gradient with weight $\beta_1$. Update the second moment as an exponential moving average of the *squared* gradient with weight $\beta_2$. Then divide each by $(1 - \beta^t)$ to correct for initialization bias. Finally, update the parameters using the corrected first moment divided by the square root of the corrected second moment.
</details>

<details>
<summary><b>Solution</b></summary>

```python
self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
m_hat = self.m[i] / (1 - self.beta1 ** self.t)
v_hat = self.v[i] / (1 - self.beta2 ** self.t)
p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)
```
</details>

---

### Comparing Your Optimizers

Train the same architecture with each of your optimizers and compare convergence speed and final accuracy.

In [ ]:
results = {}
for name, OptClass, kwargs in [
    ('SGD',          VanillaSGD,  dict(lr=0.01)),
    ('SGD+Momentum', SGDMomentum, dict(lr=0.01, momentum=0.9)),
    ('Adam',         Adam,        dict(lr=0.001)),
]:
    torch.manual_seed(0)
    model = BananaNet()
    opt = OptClass(model.parameters(), **kwargs)
    losses = train(model, opt, train_set, epochs=50)
    acc = accuracy(model)
    results[name] = {'losses': losses, 'acc': acc, 'model': model}
    print(f"{name:15s}  accuracy: {acc:.1%}")

plt.figure(figsize=(8, 4))
for name, r in results.items():
    plt.plot(r['losses'], label=f"{name} ({r['acc']:.0%})")
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Optimizer Comparison')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
xx = torch.linspace(x_val[:, 0].min() - 0.5, x_val[:, 0].max() + 0.5, 200)
yy = torch.linspace(x_val[:, 1].min() - 0.5, x_val[:, 1].max() + 0.5, 200)
gx, gy = torch.meshgrid(xx, yy, indexing='ij')
grid = torch.stack([gx.flatten(), gy.flatten()], dim=1)

for ax, (name, r) in zip(axes, results.items()):
    with torch.no_grad():
        probs = torch.sigmoid(r['model'](grid)).reshape(200, 200)
    ax.contourf(xx, yy, probs, levels=20, cmap='RdBu_r', alpha=0.6)
    ax.scatter(x_val[:, 0], x_val[:, 1], c=y_val, cmap='RdBu', s=3, alpha=0.3)
    ax.set_title(f"{name} ({r['acc']:.0%})")
    ax.axis('equal')
plt.tight_layout()
plt.show()